In [1]:
import sys

print("Python version: ", sys.version)
print("Python executable: ", sys.executable)

Python version:  3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Python executable:  /Users/timmayabi/Projects/newsfindr/.venv/bin/python


In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

#load environment variables from .env file
load_dotenv()

#verify that the Groq API exists
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Groq API key not found in environment variables.")
else:
    print("Groq API key found.")


Groq API key found.



### Observation

The Groq API key was successfully retrieved from the
local environment configuration using python-dotenv.

The API key is stored separately from the notebook
to prevent accidental exposure of sensitive credentials.

### Initializing the Groq LLM

In [3]:
# Model configuration
MODEL_NAME = "openai/gpt-oss-120b"

# Intitialize Groq LLM
llm = ChatGroq(
    model_name=MODEL_NAME,
    temperature=0,
    max_tokens=1000,
    max_retries=2,
    groq_api_key=groq_api_key
)
print("Groq LLM initialized successfully.")
print("Model:", MODEL_NAME)

Groq LLM initialized successfully.
Model: openai/gpt-oss-120b


In [5]:
# Simple query to verify Groq API connectivity
query = "What is Agentic AI? Explain in two sentences."
response = llm.invoke(query)
print("Query: ", query)
print("\nLLM response:")
print(response.content)

Query:  What is Agentic AI? Explain in two sentences.

LLM response:
Agentic AI refers to artificial intelligence systems designed to act autonomously toward specific goals, making decisions, planning, and executing actions without constant human oversight. These agents combine perception, reasoning, and learning capabilities to interact with their environment and adapt their behavior over time.


In [6]:
# Inspect token usage reported by Groq

usage = response.usage_metadata or {}

print("Input tokens:", usage.get("input_tokens", "N/A"))
print("Output tokens:", usage.get("output_tokens", "N/A"))
print("Total tokens:", usage.get("total_tokens", "N/A"))

Input tokens: 82
Output tokens: 91
Total tokens: 173



### Observations

- The Groq-hosted LLM was initialized using LangChain.
- The model successfully processed a simple natural-language query.
- A valid response was returned, confirming API connectivity.
- Token usage information was retrieved to support future
  rate-limit monitoring.

### Conclusion

The LLM is operational and ready for integration with
the SQL and news retrieval agents.


In [9]:
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, inspect

# Locate the customer database

DB_PATH = Path.cwd() / "customer.db"

#Check whether the database file exists
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database file not found at {DB_PATH}")
else:
    print(f"Database file found at {DB_PATH}")
    print("Database size:", DB_PATH.stat().st_size, "bytes")

Database file found at /Users/timmayabi/Projects/newsfindr/customer.db
Database size: 20480 bytes


In [10]:
# Connect to the customer database using SQLAlchemy
engine = create_engine(f"sqlite:///{DB_PATH}")
inspector = inspect(engine)
print("Tables in the database:", inspector.get_table_names())

# Optionally, inspect the columns of each table
for table_name in inspector.get_table_names():
    print(f"\nColumns in table '{table_name}':")
    for column in inspector.get_columns(table_name):
        print(f" - {column['name']} ({column['type']})")

Tables in the database: ['customers']

Columns in table 'customers':
 - id (INTEGER)
 - customer_id (TEXT)
 - name (TEXT)
 - email (TEXT)
 - interests (TEXT)
 - last_updated (TIMESTAMP)


In [14]:
# Verify the number of customer records
from sqlalchemy import text
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM customers"))
    customer_count = result.scalar()
print("Number of customer records:", customer_count)

Number of customer records: 15


In [16]:
from langchain_community.utilities.sql_database import SQLDatabase

# Initialize the LanchChain SQL database connection
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH}",
    include_tables=["customers"],
    sample_rows_in_table_info=0
)
#Verify accessible tables
print("Accessible tables in the database:", db.get_usable_table_names())

Accessible tables in the database: ['customers']


In [19]:
# Initialize SQLDatabaseToolkit

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Retrieve available SQL tools
sql_tools = toolkit.get_tools()

print ("SQL toolkit initialized successfully: \n")

for tool in sql_tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description[: 120]}...\n")

SQL toolkit initialized successfully: 

Tool: sql_db_query
Description: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correc...

Tool: sql_db_schema
Description: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure t...

Tool: sql_db_list_tables
Description: Input is an empty string, output is a comma-separated list of tables in the database....

Tool: sql_db_query_checker
Description: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a quer...




### Observations

- Successfully connected LangChain to the SQLite database.
- Confirmed that the `customers` table is accessible.
- Initialized the SQLDatabaseToolkit using the Groq LLM.
- Retrieved the SQL tools for schema inspection and query execution.

### Conclusion

The SQL toolkit has been initialized and is ready
for integration with an agent that retrieves customer
profiles and interests.


In [20]:

# Define the SQL Agent's behavior and constraints

SQL_AGENT_PREFIX = """
You are NewsFindr's Customer Information SQL Agent.

Your responsibility is to retrieve customer information
from a SQLite database to support personalized news retrieval.

DATABASE INFORMATION:
- Database type: SQLite
- Table: customers
- Columns:
    id (INTEGER)
    customer_id (TEXT)
    name (TEXT)
    email (TEXT)
    interests (TEXT)
    last_updated (TIMESTAMP)

INSTRUCTIONS:
1. Verify customers using their email addresses.
2. Retrieve customer_id, name, email, and interests.
3. Use the customers table only.
4. Generate valid SQLite SELECT queries.
5. Never use INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, or other database modification statements.
6. Do not invent customer information.
7. If the email is not found, clearly indicate that
   the customer does not exist.
8. Return only the information necessary to answer
   the user's request.
9. Do not expose unrelated customer records.
10. Do not execute queries unrelated to customer
    verification or customer interests.

IMPORTANT:
Always inspect the database schema before generating
queries if the schema is not already known.
Limit query results to the relevant customer.
"""

print("SQL Agent system message configured successfully.")


SQL Agent system message configured successfully.


In [21]:

from sqlalchemy import create_engine, event
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# Connect to the existing SQLite file in read-only mode
read_only_engine = create_engine(
    "sqlite:///file:newsfindr_customers"
    f"?mode=ro&uri=true",
    creator=lambda: __import__("sqlite3").connect(
        DB_PATH.resolve().as_uri() + "?mode=ro",
        uri=True
    )
)

# Restrict SQL statements to read-only operations
@event.listens_for(read_only_engine, "connect")
def enforce_read_only(dbapi_connection, connection_record):
    dbapi_connection.execute("PRAGMA query_only = ON")

# Create read-only LangChain database wrapper
readonly_db = SQLDatabase(
    read_only_engine,
    include_tables=["customers"],
    sample_rows_in_table_info=0
)

# Recreate toolkit with read-only database
readonly_toolkit = SQLDatabaseToolkit(
    db=readonly_db,
    llm=llm
)

print("Read-only SQL toolkit initialized.")
print("Available tables:", readonly_db.get_usable_table_names())


Read-only SQL toolkit initialized.
Available tables: ['customers']


In [22]:

from langchain_community.agent_toolkits.sql.base import create_sql_agent

# Initialize the SQL Agent
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=readonly_toolkit,
    agent_type="tool-calling",
    prefix=SQL_AGENT_PREFIX,
    verbose=True,
    max_iterations=8,
    agent_executor_kwargs={
        "handle_parsing_errors": True
    }
)

print("NewsFindr SQL Agent initialized successfully!")


NewsFindr SQL Agent initialized successfully!


In [23]:

print("Agent type:", type(sql_agent).__name__)
print("Number of SQL tools:", len(readonly_toolkit.get_tools()))
print("Agent initialization completed.")


Agent type: AgentExecutor
Number of SQL tools: 4
Agent initialization completed.



### Observations

- Defined a system message specifying the SQL Agent's
  responsibilities and constraints.
- Configured read-only access to the SQLite database.
- Initialized the SQL Agent using the Groq LLM and
  SQLDatabaseToolkit.
- Enabled tool-calling to support natural-language
  database queries.

### Conclusion

The NewsFindr SQL Agent has been initialized and is
ready to verify customer email addresses and retrieve
their interests.

In [24]:

from sqlalchemy import text

# Retrieve one customer email for testing
with read_only_engine.connect() as connection:
    result = connection.execute(
        text("SELECT email FROM customers LIMIT 1")
    )
    customer_email = result.scalar()

assert customer_email is not None, "No customer email found."

print("Test customer email:", customer_email)


Test customer email: alice.6eb33c45-5@gmail.com


In [25]:

# Construct the customer verification request
verification_query = f"""
Verify whether the following customer exists:

Email: {customer_email}

If the customer exists:
1. Retrieve customer_id, name, email, and interests.
2. Return the customer's details clearly.
3. Do not retrieve unrelated customer records.

If the customer does not exist:
Return "Customer not found".
"""

# Invoke the SQL Agent
customer_response = sql_agent.invoke({
    "input": verification_query
})

print("\nCustomer Verification Result:")
print(customer_response["output"])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


customers
Invoking: `sql_db_schema` with `{'table_names': 'customers'}`



CREATE TABLE customers (
	id INTEGER, 
	customer_id TEXT NOT NULL, 
	name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	interests TEXT NOT NULL, 
	last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (customer_id), 
	UNIQUE (email)
)
Invoking: `sql_db_query_checker` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;"}`


SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;
Invoking: `sql_db_query` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'alice.6eb33c45-5@gmail.com' LIMIT 1;"}`


[('6EB33C45-5', 'Alice', 'alice.6eb33c45-5@gmail.com', '["Politics", "Technology", "Business"]')]**Customer Details**

- **Customer ID:** 6EB33C45-5  

In [26]:

# Test customer verification with an unknown email
invalid_email = "nonexistent-user@example.invalid"

invalid_query = f"""
Check whether a customer exists with email:
{invalid_email}

If found, return customer_id, name, email, and interests.
Otherwise, return exactly: Customer not found.
"""

invalid_response = sql_agent.invoke({
    "input": invalid_query
})

print("\nInvalid Email Verification Result:")
print(invalid_response["output"])




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


customers
Invoking: `sql_db_schema` with `{'table_names': 'customers'}`



CREATE TABLE customers (
	id INTEGER, 
	customer_id TEXT NOT NULL, 
	name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	interests TEXT NOT NULL, 
	last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (customer_id), 
	UNIQUE (email)
)
Invoking: `sql_db_query_checker` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;"}`


SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;
Invoking: `sql_db_query` with `{'query': "SELECT customer_id, name, email, interests FROM customers WHERE email = 'nonexistent-user@example.invalid' LIMIT 1;"}`


Customer not found.

> Finished chain.

Invalid Email Verification Result:
Customer not found.


In [27]:

# Independently validate the test cases
with read_only_engine.connect() as connection:
    valid_count = connection.execute(
        text("SELECT COUNT(*) FROM customers WHERE email = :email"),
        {"email": customer_email}
    ).scalar()

    invalid_count = connection.execute(
        text("SELECT COUNT(*) FROM customers WHERE email = :email"),
        {"email": invalid_email}
    ).scalar()

assert valid_count >= 1, "Valid customer verification failed."
assert invalid_count == 0, "Invalid email unexpectedly exists."

print("Valid customer exists:", valid_count >= 1)
print("Invalid customer exists:", invalid_count > 0)
print("Database verification checks passed!")


Valid customer exists: True
Invalid customer exists: False
Database verification checks passed!



### Customer Verification Observations

- The SQL Agent successfully processed a natural-language
  customer verification request.
- The agent retrieved customer details using the email address.
- Customer information included the customer ID, name,
  email address, and registered news interests.
- The agent correctly handled an unregistered email.
- Independent database queries confirmed the test cases.

### Conclusion

The SQL Agent can verify registered customers and retrieve
their interests from the SQLite database. These interests
will be used to personalize subsequent news searches.


## 3. Interface Between SQL and Search Agents

In [28]:

from sqlalchemy import text

def get_customer_profile(email: str):
    """
    Retrieve a customer's profile using their email address.

    Returns:
        dict: Customer details if found.
        None: If the customer does not exist.
    """

    query = text("""
        SELECT customer_id, name, email, interests
        FROM customers
        WHERE LOWER(email) = LOWER(:email)
        LIMIT 1
    """)

    with read_only_engine.connect() as connection:
        result = connection.execute(
            query,
            {"email": email.strip()}
        ).mappings().first()

    return dict(result) if result else None


# Use the customer email verified in Step 3.4
customer_profile = get_customer_profile(customer_email)

assert customer_profile is not None, "Customer not found."

print("Customer profile retrieved successfully!")
print("Customer ID:", customer_profile["customer_id"])
print("Name:", customer_profile["name"])
print("Interests:", customer_profile["interests"])


Customer profile retrieved successfully!
Customer ID: 6EB33C45-5
Name: Alice
Interests: ["Politics", "Technology", "Business"]


In [31]:


import json

# Extract customer interests
raw_interests = customer_profile["interests"]

# Parse the JSON-formatted interests
customer_interests = json.loads(raw_interests)

# Clean and validate the interests
customer_interests = [
    interest.strip()
    for interest in customer_interests
    if isinstance(interest, str) and interest.strip()
]

assert customer_interests, "Customer has no registered interests."

print("Customer interests:")
for index, interest in enumerate(customer_interests, start=1):
    print(f"{index}. {interest}")



Customer interests:
1. Politics
2. Technology
3. Business


In [ ]:

# Prepare the data passed to the Search Agent
search_context = {
    "customer_id": customer_profile["customer_id"],
    "interests": customer_interests,
    "max_results": 3,
    "language": "en"
}

print("Search context prepared successfully!")

for key, value in search_context.items():
    print(f"{key}: {value}")


Search context prepared successfully!
customer_id: 6EB33C45-5
interests: ['Politics', 'Technology', 'Business']
max_results: 3
language: en



### Observations

- Retrieved a verified customer's profile from SQLite.
- Extracted and normalized the customer's news interests.
- Prepared a structured search context containing the
  customer ID, interests, language, and result limit.

### Conclusion

Customer interests are now available as structured input
for the news Search Agent. The next stage will use the
Groq LLM to generate expanded search queries.


### Expanding Customer Interests into News Search Queries

In [33]:

import json
from datetime import datetime, timezone
from langchain_core.messages import SystemMessage, HumanMessage

def expand_search_queries(interests, user_query="latest news"):
    """
    Use Groq LLM to generate precise news search queries
    based on customer interests.
    """

    current_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    system_prompt = """
    You are NewsFindr's News Search Query Expansion Agent.

    Your task is to transform customer interests into
    precise, time-sensitive news search queries.

    Rules:
    1. Generate one search query per customer interest.
    2. Focus on recent news and developments.
    3. Incorporate the user's specific request.
    4. Keep each query concise and relevant.
    5. Do not invent news events or headlines.
    6. Return ONLY a valid JSON array of strings.
    7. Do not include explanations or Markdown.
    """

    user_prompt = f"""
    Current date: {current_date}
    Customer interests: {json.dumps(interests)}
    User request: {user_query}

    Generate precise news search queries.
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    # Parse the LLM response as JSON
    try:
        queries = json.loads(response.content)
        if not isinstance(queries, list):
            raise ValueError("Expected a JSON array")

        queries = [
            q.strip() for q in queries
            if isinstance(q, str) and q.strip()
        ]

        if not queries:
            raise ValueError("No valid queries returned")

    except (json.JSONDecodeError, ValueError):
        # Fallback if the LLM returns invalid JSON
        queries = [
            f"{interest} latest news {current_date}"
            for interest in interests
        ]

    return queries[:len(interests)]


In [34]:

expanded_queries = expand_search_queries(
    interests=search_context["interests"],
    user_query="Find the latest important news"
)

print("Expanded News Search Queries:\n")

for index, query in enumerate(expanded_queries, start=1):
    print(f"{index}. {query}")


Expanded News Search Queries:

1. latest important political news October 2026
2. latest important technology news October 2026
3. latest important business news October 2026


In [35]:

assert isinstance(expanded_queries, list)
assert len(expanded_queries) == len(customer_interests)
assert all(isinstance(q, str) and q.strip()
           for q in expanded_queries)

print("Search query expansion validation passed!")
print("Total queries generated:", len(expanded_queries))


Search query expansion validation passed!
Total queries generated: 3



### Observations

- Customer interests were successfully parsed from JSON.
- The Groq LLM generated personalized news search queries.
- Search queries incorporated the customer's interests
  and the current date.
- The generated queries passed basic format validation.

### Conclusion

The query expansion component is ready to provide
personalized search terms to DuckDuckGo for retrieving
recent news articles.


In [38]:

from ddgs import DDGS
import json
import pandas as pd


In [39]:

def search_news(query: str, max_results: int = 3) -> list:
    """
    Search DuckDuckGo for recent news articles.

    Args:
        query: Expanded news search query.
        max_results: Maximum articles to retrieve.

    Returns:
        List of normalized news articles.
    """

    articles = []

    try:
        with DDGS() as ddgs:
            results = list(
                ddgs.news(
                    query=query,
                    max_results=max_results,
                    timelimit="w"
                )
            )

        for result in results:
            article = {
                "title": result.get("title", ""),
                "url": result.get("url", ""),
                "source": result.get("source", ""),
                "date": result.get("date", ""),
                "body": result.get("body", ""),
                "search_query": query
            }

            if article["title"] and article["url"]:
                articles.append(article)

    except Exception as e:
        print(f"News search failed for '{query}': {e}")

    return articles


In [40]:

# Test using the first expanded query
test_query = expanded_queries[0]

test_results = search_news(
    query=test_query,
    max_results=3
)

print(f"Search query: {test_query}")
print(f"Articles retrieved: {len(test_results)}")

for index, article in enumerate(test_results, start=1):
    print(f"\nArticle {index}")
    print("Title:", article["title"])
    print("Source:", article["source"])
    print("Date:", article["date"])
    print("URL:", article["url"])


Search query: latest important political news October 2026
Articles retrieved: 3

Article 1
Title: Latest news bulletin | October 16th, 2026 - Midday
Source: Yahoo
Date: 2026-10-16T10:00:00+00:00
URL: https://www.yahoo.com/news/world/articles/latest-news-bulletin-october-16th-100015822.html

Article 2
Title: In October 2029, will Israelis be able to say they are mentally better off than they were in October 2026?
Source: allisraelnews
Date: 2026-10-18T00:00:00+00:00
URL: https://allisraelnews.com/in-october-2029-will-israelis-be-able-to-say-they-are-mentally-better-off-than-they-were-in-october-2026

Article 3
Title: Today's Mexican Political and Economic News (Thursday, October 15, 2026)
Source: note
Date: 2026-10-15T20:22:00+00:00
URL: https://note.com/orangeman_x/n/n4d0f65b80d72?hl=en


In [41]:

all_news_results = []

for query in expanded_queries:
    print(f"\nSearching: {query}")

    results = search_news(
        query=query,
        max_results=3
    )

    print(f"Retrieved {len(results)} articles")

    all_news_results.extend(results)

print("\nTotal articles retrieved:", len(all_news_results))



Searching: latest important political news October 2026
Retrieved 3 articles

Searching: latest important technology news October 2026
Retrieved 3 articles

Searching: latest important business news October 2026
Retrieved 3 articles

Total articles retrieved: 9


In [42]:

news_df = pd.DataFrame(all_news_results)

if not news_df.empty:
    news_df = news_df.drop_duplicates(subset=["url"])

    display(
        news_df[
            ["title", "source", "date", "url", "search_query"]
        ]
    )

    print("Unique articles:", len(news_df))
else:
    print("No news articles were retrieved.")


,title,source,date,url,search_query
0,"At latest No Kings rallies, smaller crowds and...",Chicago Tribune,2026-10-11T07:09:33+00:00,https://www.chicagotribune.com/2026/10/17/no-k...,latest important political news October 2026
1,SADC representative sees Tana Forum as platfor...,Ethiopian News Agency on MSN,2026-10-11T07:09:33+00:00,https://www.msn.com/en-xl/news/other/sadc-repr...,latest important political news October 2026
2,"Why did Trump approach Putin? The ""numerical c...",note,Opinion18 hours ago,https://note.com/okmotto/n/nd1bdb23dffc0?hl=en,latest important political news October 2026
3,"Sayaka's AI News Five October 16, 2026 (Fri)",note,2026-10-16T13:23:00+00:00,https://note.com/sayasayat1999923/n/n4037d8e1c...,latest important technology news October 2026
4,"School assembly news headlines (October 14, 20...",Jagran Josh,2026-10-13T14:16:55+00:00,https://www.msn.com/en-in/news/other/school-as...,latest important technology news October 2026
5,"In October 2029, will Israelis be able to say ...",allisraelnews,2026-10-18T00:00:00+00:00,https://allisraelnews.com/in-october-2029-will...,latest important technology news October 2026
6,"Magnolia Mornings: October 14, 2026",Delta Democrat Times,2026-10-14T07:09:37+00:00,https://www.ddtonline.com/magnolia-mornings-oc...,latest important business news October 2026
7,"TomTom NV Reports Q3 Results, Highlights Narro...",Guru Focus,2026-10-15T07:09:37+00:00,https://www.gurufocus.com/news/9124885/tomtom-...,latest important business news October 2026
8,"Wayfair Coupon Codes, Promos for October 2026,...",SheKnows · via Yahoo Shopping,2026-10-14T07:09:37+00:00,https://shopping.yahoo.com/deals/articles/wayf...,latest important business news October 2026


Unique articles: 9



### Observations

- DuckDuckGo was integrated as the live news retrieval tool.
- Expanded search queries were used to retrieve recent articles.
- Search results were normalized into structured records
  containing titles, sources, dates, URLs and snippets.
- Duplicate article URLs were removed.
- Retrieved news records were displayed for inspection.

### Conclusion

The news retrieval component provides real article URLs
for the next stage of the pipeline: relevance and
source credibility filtering.
